# 02 Grad-CAM debug tren Google Colab

Notebook nay dung de xem model dang nhin vao vung nao khi doan cac class yeu. Nen chay sau khi da train xong notebook 00.

## 1. Clone hoac pull code moi nhat

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/Vo-Minh-Tri1412/cnn-food-recognition.git"
BRANCH = "codex/colab-kaggle-workflow"
PROJECT_ROOT = Path("/content/cnn-food-recognition")


def run(cmd, cwd=None, env=None):
    print("+", " ".join(str(x) for x in cmd))
    subprocess.run([str(x) for x in cmd], cwd=cwd, env=env, check=True)

if not PROJECT_ROOT.exists():
    run(["git", "clone", "-b", BRANCH, REPO_URL, PROJECT_ROOT])
else:
    run(["git", "fetch", "origin"], cwd=PROJECT_ROOT)
    run(["git", "checkout", BRANCH], cwd=PROJECT_ROOT)
    run(["git", "pull", "origin", BRANCH], cwd=PROJECT_ROOT)

os.chdir(PROJECT_ROOT)
print("PROJECT_ROOT =", PROJECT_ROOT)
print("BRANCH =", BRANCH)


## 2. Cai dependency nhe cho Grad-CAM

In [ ]:
run([sys.executable, "-m", "pip", "install", "-q", "opencv-python", "pillow", "pandas", "matplotlib"])


## 3. Mount Drive va tim model moi nhat

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/canteen_checkout")
DRIVE_RUNS_DIR = DRIVE_ROOT / "runs"
print("DRIVE_ROOT =", DRIVE_ROOT)

def newest_path(paths):
    paths = [Path(p) for p in paths if Path(p).exists()]
    if not paths:
        return None
    return max(paths, key=lambda p: p.stat().st_mtime)

model_candidates = list(DRIVE_RUNS_DIR.rglob("dish_classifier.pt"))
MODEL_PATH = newest_path(model_candidates)
if MODEL_PATH is None:
    raise FileNotFoundError("Khong tim thay dish_classifier.pt trong MyDrive/canteen_checkout/runs. Hay train notebook 00 truoc.")
print("MODEL_PATH =", MODEL_PATH)


## 4. Lay dataset classification tu Drive

Notebook se copy `classification.zip` tu Drive ve runtime roi unzip local, khong doc anh le truc tiep tren Drive.

In [ ]:
DATA_ZIP = DRIVE_ROOT / "datasets" / "classification.zip"
WORK_DATA_ROOT = Path("/content/canteen_checkout_data")
CLASSIFICATION_ROOT = WORK_DATA_ROOT / "classification"

if not DATA_ZIP.exists():
    raise FileNotFoundError(f"Khong thay dataset zip: {DATA_ZIP}")

if not CLASSIFICATION_ROOT.exists():
    WORK_DATA_ROOT.mkdir(parents=True, exist_ok=True)
    local_zip = WORK_DATA_ROOT / "classification.zip"
    shutil.copy2(DATA_ZIP, local_zip)
    shutil.unpack_archive(str(local_zip), str(WORK_DATA_ROOT))

print("CLASSIFICATION_ROOT =", CLASSIFICATION_ROOT)
print("test exists =", (CLASSIFICATION_ROOT / "test").exists())


## 5. Chon class can debug

Mac dinh uu tien cac class hay nham: canh chua, rau xao, dau hu, thit kho.

In [ ]:
CLASSES = "canh_chua_co_ca,canh_chua_khong_ca,rau_xao,canh_rau,dau_hu_sot_ca,thit_kho"
MAX_PER_CLASS = 12
GRADCAM_OUT = Path("/content/canteen_gradcam")
print("CLASSES =", CLASSES)


## 6. Chay Grad-CAM

In [ ]:
run([
    sys.executable, "scripts/24_gradcam_debug.py",
    "--model", MODEL_PATH,
    "--data", CLASSIFICATION_ROOT / "test",
    "--classes", CLASSES,
    "--max-per-class", str(MAX_PER_CLASS),
    "--out", GRADCAM_OUT,
], cwd=PROJECT_ROOT)


## 7. Xem summary va contact sheet

In [ ]:
from IPython.display import Image as IPImage, display

runs = [p for p in GRADCAM_OUT.iterdir() if p.is_dir()]
if not runs:
    raise FileNotFoundError("Khong co Grad-CAM run nao.")
LATEST_GRADCAM = max(runs, key=lambda p: p.stat().st_mtime)
summary = json.loads((LATEST_GRADCAM / "summary.json").read_text(encoding="utf-8"))
print(json.dumps(summary, ensure_ascii=False, indent=2))

for sheet in sorted(LATEST_GRADCAM.glob("contact_*.jpg")):
    print("\n", sheet.name)
    display(IPImage(filename=str(sheet), width=900))


## 8. Copy Grad-CAM ve Drive

In [ ]:
target = DRIVE_ROOT / "gradcam_runs" / LATEST_GRADCAM.name
if target.exists():
    shutil.rmtree(target)
shutil.copytree(LATEST_GRADCAM, target)
print("Da copy Grad-CAM ve Drive:", target)
